# 02 — Feature Engineering

Validação e análise das features construídas:
- Encoding de categóricas (One-Hot, Target Encoding)
- Holt-Winters: componentes e qualidade do ajuste
- Importância de amenities
- Visualização do dataset final

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

PROCESSED_PATH = Path('../data/processed')

## 1. Holt-Winters — Análise da Sazonalidade

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pathlib import Path

RAW_PATH = Path('../data/raw')
CITY = 'sao-paulo'

calendar = pd.read_csv(RAW_PATH / f'{CITY}_calendar.csv.gz', compression='gzip', parse_dates=['date'])
calendar['price_num'] = (
    calendar['price'].astype(str)
    .str.replace(r'[$,]', '', regex=True)
    .astype(float)
)
daily = calendar.groupby('date')['price_num'].median().sort_index().asfreq('D').interpolate()

# Ajuste Holt-Winters
hw_model = ExponentialSmoothing(
    daily, trend='add', seasonal='add',
    seasonal_periods=7, initialization_method='estimated'
).fit(optimized=True)

print(f'AIC: {hw_model.aic:.2f}')
print(f'Parâmetros: alpha={hw_model.params["smoothing_level"]:.3f}, '
      f'beta={hw_model.params["smoothing_trend"]:.3f}, '
      f'gamma={hw_model.params["smoothing_seasonal"]:.3f}')

In [ ]:
# Visualização dos componentes
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)

# Fitted vs Observed
sample = daily.iloc[:90]
fitted_sample = hw_model.fittedvalues.iloc[:90]
axes[0].plot(sample.index, sample.values, label='Observado', alpha=0.7)
axes[0].plot(fitted_sample.index, fitted_sample.values, label='HW Fitted', linestyle='--', color='red')
axes[0].set_title('Holt-Winters: Observado vs Ajustado (primeiros 90 dias)')
axes[0].legend()
axes[0].set_ylabel('R$')

# Componente sazonal semanal
seasonal_week = hw_model.season.iloc[:7]
days = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab', 'Dom']
axes[1].bar(days, seasonal_week.values, color='teal')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Componente Sazonal por Dia da Semana')
axes[1].set_ylabel('Ajuste de preço (R$)')

# Resíduos
residuals = daily - hw_model.fittedvalues
axes[2].plot(residuals.index, residuals.values, linewidth=0.5, color='gray')
axes[2].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[2].set_title(f'Resíduos (std={residuals.std():.2f})')
axes[2].set_ylabel('Resíduo (R$)')

plt.tight_layout()
plt.savefig(PROCESSED_PATH / 'eda_holtwinters.png', bbox_inches='tight')
plt.show()

In [ ]:
# Previsão 30 dias à frente
forecast = hw_model.forecast(30)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily.iloc[-60:].index, daily.iloc[-60:].values, label='Histórico', color='steelblue')
ax.plot(forecast.index, forecast.values, label='Previsão 30d', color='orange', linestyle='--')
ax.fill_between(forecast.index,
                forecast.values * 0.85,
                forecast.values * 1.15,
                alpha=0.2, color='orange', label='Intervalo ±15%')
ax.legend()
ax.set_title('Previsão Holt-Winters — Próximos 30 dias')
ax.set_ylabel('Preço mediano (R$)')
plt.tight_layout()
plt.show()

## 2. Target Encoding — Validação

In [ ]:
# Verificar que o target encoding não vazou informação (CV)
import pandas as pd
import numpy as np
from category_encoders import TargetEncoder
from sklearn.model_selection import KFold

listings = pd.read_csv(RAW_PATH / f'{CITY}_listings.csv.gz', compression='gzip', low_memory=False)
listings['price_num'] = (
    listings['price'].astype(str)
    .str.replace(r'[$,]', '', regex=True)
    .astype(float)
)
df = listings[(listings['price_num'] >= 10) & (listings['price_num'] <= 5000)].copy()
df['log_price'] = np.log1p(df['price_num'])
df = df.dropna(subset=['neighbourhood_cleansed', 'log_price'])

# Encoding com CV para evitar data leakage
kf = KFold(n_splits=5, shuffle=True, random_state=42)
df['neighbourhood_enc'] = 0.0

for train_idx, val_idx in kf.split(df):
    enc = TargetEncoder(smoothing=10, min_samples_leaf=5)
    df.iloc[val_idx, df.columns.get_loc('neighbourhood_enc')] = enc.fit_transform(
        df.iloc[train_idx][['neighbourhood_cleansed']],
        df.iloc[train_idx]['log_price']
    ).reindex(df.iloc[val_idx].index).squeeze().values

# Verificar correlação
corr = df['neighbourhood_enc'].corr(df['log_price'])
print(f'Correlação Target Encoding vs log_price (OOF): r={corr:.3f}')

# Top e bottom bairros
top_bot = df.groupby('neighbourhood_cleansed').agg(
    enc_mean=('neighbourhood_enc', 'mean'),
    price_median=('price_num', 'median'),
    count=('price_num', 'count')
).query('count >= 30').sort_values('enc_mean')

fig, ax = plt.subplots(figsize=(10, 5))
pd.concat([top_bot.head(10), top_bot.tail(10)])['price_median'].plot(
    kind='barh', ax=ax, color='steelblue'
)
ax.set_title('Preço Mediano — Top e Bottom 10 Bairros')
ax.set_xlabel('Preço mediano (R$)')
plt.tight_layout()
plt.show()

## 3. Dataset Final — Overview

In [ ]:
# Carregar features finais (após rodar o pipeline)
final_path = PROCESSED_PATH / 'final_features.parquet'

if final_path.exists():
    final = pd.read_parquet(final_path)
    print(f'Shape: {final.shape}')
    print(f'\nColunas por categoria:')
    
    categories = {
        'Numéricas base': [c for c in final.columns if c in ['accommodates', 'bathrooms', 'bedrooms', 'beds']],
        'One-Hot (room_type)': [c for c in final.columns if c.startswith('room_type_')],
        'Amenities': [c for c in final.columns if c.startswith('amenity_')],
        'Target Encoding': ['neighbourhood_enc'],
        'Holt-Winters': [c for c in final.columns if c.startswith('hw_')],
        'CLIP': [c for c in final.columns if c.startswith('clip_')],
        'YOLO': [c for c in final.columns if c.startswith(('yolo_', 'has_', 'object_', 'bed_count'))],
    }
    
    for cat, cols in categories.items():
        present = [c for c in cols if c in final.columns]
        print(f'  {cat}: {len(present)} features')
    
    # Nulos no dataset final
    null_pct = (final.isnull().sum() / len(final) * 100)
    if null_pct.max() > 0:
        print(f'\nColunas com nulos:')
        print(null_pct[null_pct > 0].sort_values(ascending=False))
    else:
        print('\nNenhum nulo no dataset final.')
else:
    print('Dataset final não encontrado. Rode o pipeline primeiro: make train')